In [1]:
import subprocess

import pandas as pd

import config, utils

# Globals

In [2]:
chm13v2_file_id = "t2t_chm13v2"

# Download and liftover third party data files

## Manual data retrieval

### UCSC RefSeq TSSs (GRCh38):
Adapted instructions from https://www.biostars.org/p/456388/#9494973:

Navigate to the Table Browser (http://genome.ucsc.edu/cgi-bin/hgTables) and make the following selections:
1. Under **Select dataset**:

    **clade**: Mammal

    **genome**: Humann

    **assembly**: Dec. 2013 (GRCh38/hg38)
    
    **group**: Genes and Gene Predictions
    
    **track**: NCBI RefSeq

    **table**: UCSC RefSeq (refGene)
2. Set the **region**: to “genome”
3. Click create next to “filter:”
4. On the “Filter on Fields from hg38.refGene” page, insert “cdsStart” next to **cdsEnd is**, change **ignored** to “!=” then click **submit**
5. Set the output format to “Selected fields from primary and related tables”. This will allow you to select fields of interest.
6. Set **Output filename** to "hg38_RefSeq_TSS.csv". Click **get output**
7. Select the following fields:
    **name**: Name of gene
    
    **chrom**: Reference sequence chromosome or scaffold
    
    **strand**: + or - for strand
    
    **txStart**: Left-hand end of transcription region (TSS for + strand entries)

    **txEnd**: Right-hand end of transcription region (TSS for - strand entries)
8. Click **get output**
9. Place "hg38_RefSeq_TSS.csv" in `data/raw`

## Automated data retrieval

* CTCF peak data (GRCh38): https://www.encodeproject.org/files/ENCFF797SDL/@@download/ENCFF797SDL.bed.gz
* CTCF motif data (hg19): https://compbio.mit.edu/encode-motifs/matches.txt.gz
* GM12878 expression data (GRCh38): https://www.encodeproject.org/files/ENCFF978HIY/@@download/ENCFF978HIY.tsv
* TSS location data: TODO: ADD THIS
* LAD data (GRCh38):
  * https://raw.githubusercontent.com/altemose/microDamID/refs/heads/master/DataAnalysis/cLAD_gold_final_250kb.bed
  * https://raw.githubusercontent.com/altemose/microDamID/refs/heads/master/DataAnalysis/ciLAD_gold_final_250kb.bed
* Chain files:
  * https://s3-us-west-2.amazonaws.com/human-pangenomics/T2T/CHM13/assemblies/chain/v1_nflo/grch38-chm13v2.chain
  * https://s3-us-west-2.amazonaws.com/human-pangenomics/T2T/CHM13/assemblies/chain/v1_nflo/hg19-chm13v2.chain
* Genome assembly: https://s3-us-west-2.amazonaws.com/human-pangenomics/T2T/CHM13/assemblies/analysis_set/chm13v2.0.fa.gz
* Mapping between RefSeq and Ensembl gene IDs: https://ftp.ncbi.nih.gov/gene/DATA/gene2ensembl.gz
* CpG Island locations (GRCh38): https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/cpgIslandExtUnmasked.txt.gz
  * Documentation: https://genome.ucsc.edu/cgi-bin/hgTrackUi?g=cpgIslandExt

~~* GM12878 ATAC-seq data (GRCh38): https://www.encodeproject.org/files/ENCFF748UZH/@@download/ENCFF748UZH.bed.gz~~

In [ ]:
t2t_reference = utils.save_file_from_url("https://s3-us-west-2.amazonaws.com/human-pangenomics/T2T/CHM13/assemblies/analysis_set/chm13v2.0.fa.gz")
_ = subprocess.run(["samtools", "faidx", t2t_reference])

In [29]:
grch38_chm13v2_chain_file = utils.save_file_from_url("https://s3-us-west-2.amazonaws.com/human-pangenomics/T2T/CHM13/assemblies/chain/v1_nflo/grch38-chm13v2.chain")
hg19_chm13v2_chain_file = utils.save_file_from_url("https://s3-us-west-2.amazonaws.com/human-pangenomics/T2T/CHM13/assemblies/chain/v1_nflo/hg19-chm13v2.chain")

In [5]:
gene2ensembl_file = utils.save_file_from_url("https://ftp.ncbi.nih.gov/gene/DATA/gene2ensembl.gz")
gene2ensembl_human_file = gene2ensembl_file.with_suffix(".human")
# Save a file mapping human (taxon 9606) NCBI ids to Ensembl ids, retaining the header row
with gene2ensembl_human_file.open("w") as fp:
    subprocess.run(
        ["awk", "NR==1 || $1 == 9606", gene2ensembl_file],
        stdout=fp, encoding="utf-8"
    )
print(gene2ensembl_human_file.relative_to(config.working_dir))

data/raw/gene2ensembl.human


In [9]:
cLAD_gold_bed = utils.lift_over(
    utils.save_file_from_url("https://raw.githubusercontent.com/altemose/microDamID/refs/heads/master/DataAnalysis/cLAD_gold_final_250kb.bed"),
    "bed",
    grch38_chm13v2_chain_file,
    chm13v2_file_id
)

lifted file: data/processed/cLAD_gold_final_250kb.t2t_chm13v2.bed
original: 1200; lifted: 1189; percent: 0.9908333333333333
total unmapped: 11
#Split in new
 11


Reading liftover chains
Mapping coordinates


In [14]:
gm12878_expression_file = utils.save_file_from_url("https://www.encodeproject.org/files/ENCFF978HIY/@@download/ENCFF978HIY.tsv")
print(gm12878_expression_file.relative_to(config.working_dir))

PosixPath('data/raw/ENCFF978HIY.tsv')

In [10]:
ciLAD_gold_bed = utils.lift_over(
    utils.save_file_from_url("https://raw.githubusercontent.com/altemose/microDamID/refs/heads/master/DataAnalysis/ciLAD_gold_final_250kb.bed"),
    "bed",
    grch38_chm13v2_chain_file,
    chm13v2_file_id
)

lifted file: data/processed/ciLAD_gold_final_250kb.t2t_chm13v2.bed
original: 1200; lifted: 1171; percent: 0.9758333333333333
total unmapped: 29
#Split in new
 27
#Partially deleted in new
 2


Reading liftover chains
Mapping coordinates


In [ ]:
# gm12878_atac_peak_file = utils.lift_over(
#     utils.save_file_from_url("https://www.encodeproject.org/files/ENCFF748UZH/@@download/ENCFF748UZH.bed.gz"),
#     "bed",
#     grch38_chm13v2_chain_file,
#     chm13v2_file_id
# )

Reading liftover chains
Mapping coordinates


lifted file: data/processed/ENCFF748UZH.t2t_chm13v2.bed
original: 277999; lifted: 277147; percent: 0.9969352407742474
total unmapped: 852
#Deleted in new
 526
#Partially deleted in new
 304
#Split in new
 22


In [4]:
gm12878_ctcf_chip_peak_bed = utils.lift_over(
    utils.save_file_from_url("https://www.encodeproject.org/files/ENCFF797SDL/@@download/ENCFF797SDL.bed.gz"),
    "bed",
    grch38_chm13v2_chain_file,
    chm13v2_file_id
)

Reading liftover chains
Mapping coordinates


lifted file: data/processed/ENCFF797SDL.t2t_chm13v2.bed
original: 41952; lifted: 41863; percent: 0.9978785278413425
total unmapped: 89
#Partially deleted in new
 69
#Deleted in new
 20


In [5]:
ctcf_motif_txt = utils.save_file_from_url("https://compbio.mit.edu/encode-motifs/matches.txt.gz")
ctcf_motif_txt = ctcf_motif_txt.rename(ctcf_motif_txt.with_suffix(".ctcf_motifs.txt"))

ctcf_motif_table = pd.read_csv(ctcf_motif_txt, sep=" ", header=None, names=["name", "chrom", "chromStart", "chromEnd", "strand"])
ctcf_motif_table = ctcf_motif_table.drop_duplicates(subset=["chrom", "chromStart", "chromEnd"])
ctcf_motif_table["name"] = "."
ctcf_motif_table["strand"] = "."
ctcf_motif_table["score"] = "."

utils.write_bed_file(ctcf_motif_table[["chrom", "chromStart", "chromEnd", "name", "score", "strand"]], config.processed_data_dir / ctcf_motif_txt.with_suffix(".bed").name)

ctcf_motif_bed = utils.lift_over(
    config.processed_data_dir / "matches.ctcf_motifs.bed",
    "bed",
    hg19_chm13v2_chain_file,
    chm13v2_file_id
)

Reading liftover chains
Mapping coordinates


lifted file: data/processed/matches.ctcf_motifs.t2t_chm13v2.bed
original: 97667053; lifted: 96978496; percent: 0.9929499562150196
total unmapped: 688557
#Deleted in new
 498403
#Partially deleted in new
 190104
#Split in new
 50


In [16]:
hg38_cpg_island_bed = config.processed_data_dir / "cpgIslands.bed"

cpg_island_txt = utils.save_file_from_url("https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/cpgIslandExtUnmasked.txt.gz")

cpg_island_table = pd.read_csv(cpg_island_txt, sep="\t", header=None, names=["bin", "chrom", "chromStart", "chromEnd", "name", "length", "cpgNum", "gcNum", "perCpg", "perGc", "obsExp"])
cpg_island_table["strand"] = "."
cpg_island_table["name"] = cpg_island_table["name"].str.replace(" ", "")
cpg_island_table = cpg_island_table[["chrom", "chromStart", "chromEnd", "name", "cpgNum", "strand", "length", "gcNum", "perCpg", "perGc", "obsExp"]]
cpg_island_table.to_csv(hg38_cpg_island_bed, sep="\t", header=None, index=None)
cpg_island_bed = utils.lift_over(
    hg38_cpg_island_bed,
    "bed",
    grch38_chm13v2_chain_file,
    chm13v2_file_id
)

Reading liftover chains
Mapping coordinates


lifted file: data/processed/cpgIslands.t2t_chm13v2.bed
original: 55149; lifted: 49633; percent: 0.8999800540354312
total unmapped: 5516
#Deleted in new
 4765
#Partially deleted in new
 739
#Split in new
 12


In [ ]:
# temp_file = config.processed_data_dir / "TEMP.bed"
# with temp_file.open("w") as fp:
#     subprocess.run(
#         ["sort", "-k1,1", "-k2,2n", cpg_island_bed],
#         stdout=fp, encoding="utf-8"
#     )
# temp_file.rename(cpg_island_bed)

PosixPath('/Users/jeremy/devspace/multimodal-dimelo/data/processed/cpgIslands.t2t_chm13v2.bed')

In [ ]:
# test = pd.read_csv(cpg_island_bed, sep="\t", header=None)
# test[0].unique()

array(['chr1', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15',
       'chr16', 'chr17', 'chr18', 'chr19', 'chr2', 'chr20', 'chr21',
       'chr22', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9',
       'chrX', 'chrY'], dtype=object)

# OPTIONAL: Redefine paths to avoid redownloading when rerunning later
TODO: Delete this

Some of these are actually generated in later cells, but this will all be deleted later so don't worry about it.

In [ ]:
gm12878_ctcf_chip_peak_bed = config.processed_data_dir / "ENCFF797SDL.t2t_chm13v2.bed"
ctcf_motif_bed = config.processed_data_dir / "matches.ctcf_motifs.t2t_chm13v2.bed"
t2t_reference = config.raw_data_dir / "chm13v2.0.fa"
hg38_refseq_tss = config.raw_data_dir / "hg38_RefSeq_TSS.csv"
gene2ensembl_human_file = config.raw_data_dir / "gene2ensembl.human"
gm12878_expression_file = config.raw_data_dir / "ENCFF978HIY.tsv"
grch38_chm13v2_chain_file = config.raw_data_dir / "grch38-chm13v2.chain"
gm12878_ranked_TSS_bed = config.processed_data_dir / "tss.gm12878_ranked.t2t_chm13v2.bed"
cpg_island_bed = config.processed_data_dir / "cpgIslands.t2t_chm13v2.bed"

# Postprocessing
Perform any bespoke processing to produce final files for analysis and plotting

## TSSs
Rank TSSs by GM12878 expression

TODO: I have stripped some extra work out of Annie's blueprints. I get a barely smaller number of TSSs out the other end (56386 from my version of hers, 55870 from mine; 99% of them). I should work to understand where these went, why they went, and whether I care.

NOTE: Remember that I have fixed the txStart/txEnd bug though, so results may look superficially different

NOTE: I changed how I'm downloading the RefSeq file too so the name column no longer seemed to come with version information. But I could be wrong about that.

In [31]:
55870/56386

0.9908487922533963

In [14]:
tss_table = pd.read_csv(hg38_refseq_tss, sep="\t", skiprows=1).rename(columns={"#name": "name"})
# Filter out non-standard chromosomes
tss_table = tss_table[tss_table["chrom"].str.split("_").apply(len) == 1]
# Get correct TSSs based on strand (note that NCBI columns are more accurately left-hand and right-hand window extents)
tss_table["tss"] = tss_table.apply(lambda row: row["txStart"] if row["strand"] == "+" else row["txEnd"], axis=1)
tss_table.head()

,name,chrom,strand,txStart,txEnd,tss
0,NM_000299,chr1,+,201283451,201332993,201283451
1,NM_001276351,chr1,-,67092165,67134970,67134970
2,NM_001005337,chr1,+,201283505,201332989,201283505
3,NM_001276352,chr1,-,67092165,67134970,67134970
4,NR_075077,chr1,-,67092165,67134970,67134970


In [15]:
gene2ensembl_table = pd.read_csv(gene2ensembl_human_file, sep="\t").rename(columns={"#tax_id": "tax_id"})
gene2ensembl_table['RNA_nucleotide_accession'] = gene2ensembl_table["RNA_nucleotide_accession.version"].str.split('.').str[0]
gene2ensembl_table.head()

,tax_id,GeneID,Ensembl_gene_identifier,RNA_nucleotide_accession.version,Ensembl_rna_identifier,protein_accession.version,Ensembl_protein_identifier,RNA_nucleotide_accession
0,9606,1,ENSG00000121410,NM_130786.4,ENST00000263100.8,NP_570602.2,ENSP00000263100.2,NM_130786
1,9606,2,ENSG00000175899,NM_000014.6,ENST00000318602.12,NP_000005.3,ENSP00000323929.8,NM_000014
2,9606,2,ENSG00000175899,NM_001347423.2,ENST00000891824.1,NP_001334352.2,ENSP00000561883.1,NM_001347423
3,9606,9,ENSG00000171428,NM_000662.8,ENST00000307719.9,NP_000653.3,ENSP00000307218.4,NM_000662
4,9606,9,ENSG00000171428,NM_001160171.4,ENST00000518029.5,NP_001153643.1,ENSP00000428270.1,NM_001160171


In [16]:
expression_table = pd.read_csv(gm12878_expression_file, sep="\t")
expression_table['ensembl_basename'] = expression_table['gene_id'].str.split('.').str[0]
expression_table.iloc[1000:1005]

,gene_id,transcript_id(s),length,effective_length,expected_count,TPM,FPKM,posterior_mean_count,posterior_standard_deviation_of_count,pme_TPM,pme_FPKM,TPM_ci_lower_bound,TPM_ci_upper_bound,TPM_coefficient_of_quartile_variation,FPKM_ci_lower_bound,FPKM_ci_upper_bound,FPKM_coefficient_of_quartile_variation,ensembl_basename
1000,ENSG00000014641.17,"ENST00000233114.12,ENST00000409476.5,ENST00000...",1151.06,1052.06,16430.91,579.15,463.97,16430.92,0.27,556.10,463.32,543.962000,568.56200,0.007651,453.444000,473.930000,0.007652,ENSG00000014641
1001,ENSG00000014824.13,"ENST00000264451.11,ENST00000505523.1,ENST00000...",2948.73,2849.73,4162.00,54.16,43.39,4162.00,0.00,52.84,44.02,49.586000,56.03360,0.021264,41.378300,46.749600,0.021266,ENSG00000014824
1002,ENSG00000014914.20,"ENST00000369140.7,ENST00000439741.3,ENST000004...",1725.61,1626.61,23.00,0.52,0.42,23.00,0.00,0.79,0.65,0.475958,1.13418,0.144302,0.387697,0.936449,0.144390,ENSG00000014914
1003,ENSG00000014919.12,"ENST00000016171.5,ENST00000370483.9,ENST000004...",2507.43,2408.43,2074.80,31.95,25.59,2073.98,4.13,30.75,25.62,29.319000,32.17600,0.016029,24.457600,26.835900,0.016031,ENSG00000014919
1004,ENSG00000015133.18,"ENST00000331194.8,ENST00000334448.5,ENST000003...",2796.98,2697.98,1814.00,24.93,19.97,1814.00,0.00,24.95,20.79,22.200600,27.80500,0.039059,18.509800,23.182500,0.039058,ENSG00000015133


Merge the three tables

In [21]:
merged_table = tss_table.merge(
    gene2ensembl_table, left_on='name', right_on='RNA_nucleotide_accession'
).merge(
    expression_table, left_on='Ensembl_gene_identifier', right_on='ensembl_basename'
)
merged_table.head()

,name,chrom,strand,txStart,txEnd,tss,tax_id,GeneID,Ensembl_gene_identifier,RNA_nucleotide_accession.version,...,posterior_standard_deviation_of_count,pme_TPM,pme_FPKM,TPM_ci_lower_bound,TPM_ci_upper_bound,TPM_coefficient_of_quartile_variation,FPKM_ci_lower_bound,FPKM_ci_upper_bound,FPKM_coefficient_of_quartile_variation,ensembl_basename
0,NM_000299,chr1,+,201283451,201332993,201283451,9606,5317,ENSG00000081277,NM_000299.4,...,0.00,0.14,0.12,0.018213,0.299457,0.379752,0.015151,0.249586,0.379790,ENSG00000081277
1,NM_001276351,chr1,-,67092165,67134970,67134970,9606,400757,ENSG00000203963,NM_001276351.2,...,0.00,0.19,0.16,0.055834,0.346380,0.274210,0.046528,0.288688,0.274145,ENSG00000203963
2,NM_001005337,chr1,+,201283505,201332989,201283505,9606,5317,ENSG00000081277,NM_001005337.3,...,0.00,0.14,0.12,0.018213,0.299457,0.379752,0.015151,0.249586,0.379790,ENSG00000081277
3,NM_001276352,chr1,-,67092165,67134970,67134970,9606,400757,ENSG00000203963,NM_001276352.2,...,0.00,0.19,0.16,0.055834,0.346380,0.274210,0.046528,0.288688,0.274145,ENSG00000203963
4,NM_001042681,chr1,-,8352403,8817640,8817640,9606,473,ENSG00000142599,NM_001042681.2,...,0.63,16.52,13.76,13.853700,19.293200,0.057277,11.558400,16.090500,0.057364,ENSG00000142599


Convert to a standard BED6 file. Store both the RefSeq and Ensembl ids as the entry names. Deduplicate by position and sort by TPM

In [24]:
tss_bed_table = merged_table.assign(name=merged_table["name"] + ";" + merged_table["ensembl_basename"])[["chrom", "tss", "tss", "name", "TPM", "strand"]]
tss_bed_table.columns = ["chrom", "chromStart", "chromEnd", "name", "score", "strand"]
tss_bed_table["chromEnd"] += 1
tss_bed_table = tss_bed_table.drop_duplicates(subset=["chrom", "chromStart", "chromEnd"]).sort_values(by="score", ascending=False)
tss_bed_table.head()

,chrom,chromStart,chromEnd,name,score,strand
27074,chrX,12975109,12975110,NM_021109;ENSG00000205542,27812.28,+
21211,chr7,5530601,5530602,NM_001101;ENSG00000075624,23838.43,-
36399,chr12,6535263,6535264,NM_001256799;ENSG00000111640,6774.57,+
36395,chr12,6534516,6534517,NM_001357943;ENSG00000111640,6774.57,+
41994,chr15,44711516,44711517,NM_004048;ENSG00000166710,6284.36,+


In [30]:
gm12878_ranked_TSS_hg38_bed = config.processed_data_dir / "tss.gm12878_ranked.bed"
utils.write_bed_file(tss_bed_table, gm12878_ranked_TSS_hg38_bed)
gm12878_ranked_TSS_bed = utils.lift_over(
    gm12878_ranked_TSS_hg38_bed,
    "bed",
    grch38_chm13v2_chain_file,
    chm13v2_file_id
)

Reading liftover chains
Mapping coordinates


lifted file: data/processed/tss.gm12878_ranked.t2t_chm13v2.bed
original: 35137; lifted: 34951; percent: 0.9947064348123061
total unmapped: 186
#Deleted in new
 186


## LMNB1

In [ ]:
# NOTE: Current method of avoiding duplicates results in a semi-random selection of low-quartile TSSs; consider improving this
utils.split_bed_by_quantiles(
    utils.load_bed_file(gm12878_ranked_TSS_bed),
    "score",
    q=4,
    dataset_id=gm12878_ranked_TSS_bed.with_suffix(".quartiles").name,
    avoid_duplicates=True
)

In [18]:
# NOTE: Matches window size set in LMNB1_analysis.ipynb
accessibility_on_target_window_size = 500
accessibility_off_target_window_size = 500
cpg_tss_proximity_range = 1000

gm12878_top_TSS_bed = config.processed_data_dir / "tss.gm12878_ranked.t2t_chm13v2.quartiles" / "q4.bed"
gm12878_bottom_TSS_bed = config.processed_data_dir / "tss.gm12878_ranked.t2t_chm13v2.quartiles" / "q1.bed"

Define on-target accessibility sites as a window around high-expression TSSs

In [24]:
gm12878_accessibility_on_target_file = config.processed_data_dir / "gm12878_accessibility.on_target.bed"
with gm12878_accessibility_on_target_file.open("w") as fp:
    subprocess.run(
        [config.user_config["executables"]["bedtools_exe"], "slop", "-i", gm12878_top_TSS_bed, "-g", t2t_reference.with_suffix(".fa.fai"), "-b", str(accessibility_on_target_window_size)],
        stdout=fp, encoding="utf-8"
    )

Define off-target accessibility sites as flanking regions around these strong TSSs

In [25]:
gm12878_accessibility_off_target_file = config.processed_data_dir / "gm12878_accessibility.off_target.bed"
with gm12878_accessibility_off_target_file.open("w") as fp:
    subprocess.run(
        [config.user_config["executables"]["bedtools_exe"], "flank", "-i", gm12878_accessibility_on_target_file, "-g", t2t_reference.with_suffix(".fa.fai"), "-b", str(accessibility_off_target_window_size)],
        stdout=fp, encoding="utf-8"
    )

Define on- and off-target CpG sites as CpG islands within a given proximity of inactive and active TSSs, respectively.

In [26]:
gm12878_cpg_on_target_file = config.processed_data_dir / "gm12878_cpg.on_target.bed"
gm12878_cpg_off_target_file = config.processed_data_dir / "gm12878_cpg.off_target.bed"

# Expectation is to see high CpG close to low expression TSSs, and low CpG close to high expression TSSs
for target_file, tss_file in zip([gm12878_cpg_on_target_file, gm12878_cpg_off_target_file], [gm12878_bottom_TSS_bed, gm12878_top_TSS_bed]):
    with target_file.open("w") as fp:
        subprocess.run(
            [config.user_config["executables"]["bedtools_exe"], "window", "-u", "-w", str(cpg_tss_proximity_range), "-a", cpg_island_bed, "-b", tss_file],
            stdout=fp, encoding="utf-8"
        )

## CTCF

In [33]:
ctcf_n_top_peaks = 3000
# NOTE: Matches window size set in CTCF_analysis.ipynb
ctcf_on_target_window_size = 200
ctcf_off_target_window_size = 1300

Select strong CTCF ChIP-seq peaks which overlap with a known CTCF motif, then find their center points.

In [ ]:
gm12878_CTCF_known_locations_file = config.processed_data_dir / "gm12878_ctcf_chip_intersect_motif.top3k.bed"
with gm12878_CTCF_known_locations_file.open("w") as fp:
    intersect_cmd = subprocess.Popen(
        [config.user_config["executables"]["bedtools_exe"], "intersect", "-u", "-a", gm12878_ctcf_chip_peak_bed, "-b", ctcf_motif_bed],
        stdout=subprocess.PIPE
    )
    sort_command = subprocess.Popen(
        ["sort", "-k", "7,7nr"],
        stdin=intersect_cmd.stdout, stdout=subprocess.PIPE
    )
    head_command = subprocess.Popen(
        ["head", "-n", str(ctcf_n_top_peaks)],
        stdin=sort_command.stdout, stdout=fp, encoding="utf-8"
    )
    head_command.communicate()

known_location_table = utils.load_encode_narrowpeak_bed(gm12878_CTCF_known_locations_file)
known_location_table["chromStart"] = known_location_table["chromStart"] + ((known_location_table["chromEnd"] - known_location_table["chromStart"]) // 2)
known_location_table["chromEnd"] = known_location_table["chromStart"] + 1
utils.write_bed_file(known_location_table, gm12878_CTCF_known_locations_file)

print(f"output file: {gm12878_CTCF_known_locations_file.relative_to(config.working_dir)}")

output file: data/processed/gm12878_ctcf_chip_intersect_motif.top3k.bed


Define on-target CTCF sites as a window around each known location center point.

In [ ]:
gm12878_CTCF_on_target_file = config.processed_data_dir / "gm12878_ctcf.on_target.bed"
with gm12878_CTCF_on_target_file.open("w") as fp:
    subprocess.run(
        [config.user_config["executables"]["bedtools_exe"], "slop", "-i", gm12878_CTCF_known_locations_file, "-g", t2t_reference.with_suffix(".fa.fai"), "-b", str(ctcf_on_target_window_size)],
        stdout=fp, encoding="utf-8"
    )

Define off-target CTCF sites as flanking regions around these strong known CTCF locations

In [35]:
gm12878_CTCF_off_target_file = config.processed_data_dir / "gm12878_ctcf.off_target.bed"
with gm12878_CTCF_off_target_file.open("w") as fp:
    subprocess.run(
        [config.user_config["executables"]["bedtools_exe"], "flank", "-i", gm12878_CTCF_on_target_file, "-g", t2t_reference.with_suffix(".fa.fai"), "-b", str(ctcf_off_target_window_size)],
        stdout=fp, encoding="utf-8"
    )